<a href="https://colab.research.google.com/github/weaamasad99/CloudComputingWolf/blob/main/TUT5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from collections import defaultdict

class WikiSearchEngine:
    def __init__(self):
        """Initialize the search engine"""
        self.base_url = "https://en.wikipedia.org/w/api.php"
        self.pages = []
        self.word_locations = defaultdict(list)  # word -> [(page_id, frequency), ...]
        self.stop_words = {'a', 'an', 'the', 'and', 'or', 'in', 'on', 'at', 'to', 'for', 'of', 'with'}
        return False


In [ ]:
def fetch_wiki_pages(self, topic, num_pages=5):
        """Fetch Wikipedia pages for given topic"""
        search_params = {
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": topic,
            "srlimit": num_pages
        }


In [ ]:
try:
            response = requests.get(self.base_url, params=search_params)
            search_results = response.json()['query']['search']

            for result in search_results:
                content_params = {
                    "action": "query",
                    "format": "json",
                    "prop": "extracts|info",
                    "pageids": result['pageid'],
                    "inprop": "url",
                    "explaintext": True
                }
            content_response = requests.get(self.base_url, params=content_params)
            page_data = content_response.json()['query']['pages'][str(result['pageid'])]

            self.pages.append({
                    'id': result['pageid'],
                    'title': page_data['title'],
                    'url': page_data.get('fullurl', f"https://en.wikipedia.org/?curid={result['pageid']}"),
                    'content': page_data['extract']
                })
            print(f"Retrieved: {page_data['title']}")


except Exception as e:
            print(f"Error fetching pages: {str(e)}")


In [ ]:
def build_index(self):
        """Build a simple word location index"""
        self.word_locations.clear()

        # Process each page
        for page in self.pages:
            # Get all words from content
            words = re.findall(r'\w+', page['content'].lower())

            # Count word frequencies
            word_counts = defaultdict(int)
            for word in words:
                if word not in self.stop_words:
                    word_counts[word] += 1

            # Add to index with page information
            for word, count in word_counts.items():
                self.word_locations[word].append((page['id'], count))


In [ ]:
def search(self, query, num_results=5):
        """Search pages using simple word frequency ranking.
        Ranks pages based on:1. Number of query words found in the page
        2. Total frequency of query words  """
        # Get query words
        query_words = [word.lower() for word in re.findall(r'\w+', query)
                      if word.lower() not in self.stop_words]
        if not query_words:
            return []

        # Calculate scores for each page
        page_scores = defaultdict(lambda: {'matches': 0, 'total_freq': 0})

        # For each query word
        for word in query_words:
            # Find pages containing this word
            for page_id, freq in self.word_locations.get(word, []):
                page_scores[page_id]['matches'] += 1
        # Convert to list and sort
        ranked_results = [
            (page_id, scores['matches'], scores['total_freq'])
            for page_id, scores in page_scores.items()
        ]
    # Sort by number of matching words first, then by total frequency
        ranked_results.sort(key=lambda x: (x[1], x[2]), reverse=True)
        # Format results
        results = []
        for page_id, matches, total_freq in ranked_results[:num_results]:
            page = next(p for p in self.pages if p['id'] == page_id)
            # Find the first matching word context
            context = self.get_context(page['content'], query_words)
            results.append({
                'title': page['title'],
                'url': page['url'],
                'matching_words': matches,
                'total_frequency': total_freq,
                'context': context
            })
        return results

## Wiki Search Engine for Plant Diseases

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
from collections import defaultdict

class WikiSearchEngine:
    def __init__(self):
        """Initialize the search engine"""
        self.base_url = "https://en.wikipedia.org/w/api.php"
        self.pages = []
        self.word_locations = defaultdict(list)  # word -> [(page_id, frequency), ...]
        self.stop_words = {'a', 'an', 'the', 'and', 'or', 'in', 'on', 'at', 'to', 'for', 'of', 'with'}

    def fetch_wiki_pages(self, topic, num_pages=5):
        """Fetch Wikipedia pages for given topic"""
        search_params = {
            "action": "query",
            "format": "json",
            "list": "search",
            "srsearch": topic,
            "srlimit": num_pages
        }
        headers = {
            'User-Agent': 'ColabWikiSearchEngine/1.0 (https://colab.research.google.com; user@example.com)'
        }
        try:
            response = requests.get(self.base_url, params=search_params, headers=headers)
            response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
            search_results = response.json()['query']['search']

            for result in search_results:
                # Fetch content for each page found
                content_params = {
                    "action": "query",
                    "format": "json",
                    "prop": "extracts|info",
                    "pageids": result['pageid'],
                    "inprop": "url",
                    "explaintext": True
                }
                content_response = requests.get(self.base_url, params=content_params, headers=headers)
                content_response.raise_for_status()
                page_data = content_response.json()['query']['pages'][str(result['pageid'])]

                self.pages.append({
                    'id': result['pageid'],
                    'title': page_data['title'],
                    'url': page_data.get('fullurl', f"https://en.wikipedia.org/?curid={result['pageid']}"),
                    'content': page_data['extract']
                })
                print(f"Retrieved: {page_data['title']}")

        except requests.exceptions.RequestException as e:
            print(f"Error fetching pages: {e}")
        except KeyError as e:
            print(f"Error parsing Wikipedia API response: {e}. Response might be incomplete or malformed.")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

    def build_index(self):
        """Build a simple word location index"""
        self.word_locations.clear()

        # Process each page
        for page in self.pages:
            # Get all words from content
            words = re.findall(r'\w+', page['content'].lower())

            # Count word frequencies
            word_counts = defaultdict(int)
            for word in words:
                if word not in self.stop_words:
                    word_counts[word] += 1

            # Add to index with page information
            for word, count in word_counts.items():
                self.word_locations[word].append((page['id'], count))

    def get_context(self, text, query_words, context_len=50):
        """Extracts a snippet of text around the first occurrence of a query word."""
        lower_text = text.lower()
        for q_word in query_words:
            match = re.search(r'\b' + re.escape(q_word) + r'\b', lower_text)
            if match:
                start = max(0, match.start() - context_len)
                end = min(len(text), match.end() + context_len)
                return text[start:end] + "..."
        return text[:context_len*2] + "..." # Return beginning of text if no query word found

    def search(self, query, num_results=5):
        """Search pages using simple word frequency ranking.
        Ranks pages based on:1. Number of query words found in the page
        2. Total frequency of query words  """
        # Get query words
        query_words = [word.lower() for word in re.findall(r'\w+', query)
                      if word.lower() not in self.stop_words]
        if not query_words:
            return []

        # Calculate scores for each page
        page_scores = defaultdict(lambda: {'matches': 0, 'total_freq': 0})

        # For each query word
        for word in query_words:
            # Find pages containing this word
            for page_id, freq in self.word_locations.get(word, []):
                page_scores[page_id]['matches'] += 1
                page_scores[page_id]['total_freq'] += freq # Accumulate total frequency

        # Convert to list and sort
        ranked_results = [
            (page_id, scores['matches'], scores['total_freq'])
            for page_id, scores in page_scores.items()
        ]
        # Sort by number of matching words first, then by total frequency
        ranked_results.sort(key=lambda x: (x[1], x[2]), reverse=True)
        # Format results
        results = []
        for page_id, matches, total_freq in ranked_results[:num_results]:
            page = next(p for p in self.pages if p['id'] == page_id)
            # Find the first matching word context
            context = self.get_context(page['content'], query_words)
            results.append({
                'title': page['title'],
                'url': page['url'],
                'matching_words': matches,
                'total_frequency': total_freq,
                'context': context
            })
        return results

### Demonstration of WikiSearchEngine

In [ ]:
# Initialize the search engine
engine = WikiSearchEngine()

# Fetch pages related to 'plant disease identification'
print("Fetching Wikipedia pages...")
engine.fetch_wiki_pages("plant disease identification", num_pages=5)

# Build the index
print("Building index...")
engine.build_index()

# Define the 10 significant words as requested
significant_words = [
    "fungus", "bacteria", "virus", "pathogen", "symptoms",
    "diagnosis", "control", "resistance", "treatment", "crop"
]

# Search for each significant word and display results
print("\nSearching for significant words:")
all_word_counts = defaultdict(int)

for word in significant_words:
    print(f"\n--- Search results for '{word}' ---")
    results = engine.search(word, num_results=3) # Get top 3 results for each word
    if results:
        for result in results:
            print(f"Title: {result['title']}")
            print(f"URL: {result['url']}")
            print(f"Matching words: {result['matching_words']}, Total frequency: {result['total_frequency']}")
            print(f"Context: {result['context']}\n")
    else:
        print(f"No results found for '{word}'.")

    # Aggregate word counts across all fetched pages
    for page_id, freq in engine.word_locations.get(word, []):
        all_word_counts[word] += freq

### Visualization of Word Frequencies

In [ ]:
import matplotlib.pyplot as plt

# Convert to a format suitable for plotting
words = list(all_word_counts.keys())
counts = list(all_word_counts.values())

if words:
    plt.figure(figsize=(12, 6))
    plt.bar(words, counts, color='skyblue')
    plt.xlabel('Significant Words')
    plt.ylabel('Total Occurrences Across Fetched Pages')
    plt.title('Frequency of Significant Words in Wikipedia Pages on Plant Disease Identification')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No significant word data to plot.")